# Financial Opportunity Analysis

**Program purpose:**<br>
This notebook will estimate the financial opportunity for CAR024 by combining:

1. Model outputs
Predicted probability that CAR024 should be present for each encounter.
2. Operating threshold
The validation-selected FLAML threshold: 0.23.
3. Financial benchmark data
DRG-level median charge differences between encounters where CAR024 was present vs absent.<br>

**Goal:**<br>
Identify admissions where CAR024 was not recorded, but the model says it was likely enough to warrant review.<br>

Those flagged encounters become the estimated review queue for potential missed billing opportunity.<br>

Because the dataset is national, the analysis focuses on case-level opportunity and review prioritization rather than estimating total national recoverable revenue.

**General Flow:**<br>
load data → join model features → score admissions → validate scoring → define opportunity cases → estimate case-level opportunity → rank cases for review.

### Imports and Configuration

In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import recall_score, precision_score
from datetime import datetime
import os

In [5]:
# Working directory
working_dir='/Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone/Models/'

# Identify target CCSR to include in this analysis
target_ccsr_val = "CAR024"


financial_input_file_nm='financial_opportunity_model/financial_opportunity_input_admission.csv'


### Load Financial and Model Data

In [6]:
# Load Financial Data
df = pd.read_csv(working_dir + financial_input_file_nm)
# Filter on only the CCSR we want
filtered_df =df[df["target_ccsr"]==target_ccsr_val].copy()
# Be sure DRG Code is a string
filtered_df["drg_cd"] = filtered_df["drg_cd"].astype(str)

**Load Final CAR024 Model**

This section loads the saved FLAML CAR024 model artifact from the diagnosis-procedure modeling notebook. <br>
The artifact includes the fitted model, validation-selected operating threshold, target name, and feature list required for scoring.

In [7]:
# Import desired diagnosis_procedure model
dx_pr_model_nm='diagnosis_procedure_model/flaml_car024_model.pkl'
model_artifacts=joblib.load(working_dir + dx_pr_model_nm)

In [8]:
# Extract Model Components
model=model_artifacts["model"]
threshold=model_artifacts['threshold']
features=model_artifacts['features']
target=model_artifacts['target']

In [9]:
# Confirm Model Components
print("Target:",target)
print("Threshold:",threshold)
print("Number of Features:", len(features))

Target: target_prccsr_car024
Threshold: 0.23
Number of Features: 31


In [10]:
# Import data the model was trained on
model_input_file_nm='feasibility_analysis/feasibility_model_input.csv'
data_df = pd.read_csv(working_dir + model_input_file_nm)

In [11]:
# Recreate changes to model data
# Encode Gender
data_df['gender_female'] = (data_df['gender'] == 'F').astype(int)

# Convert DRG to string
data_df['drg_cd'] = data_df['drg_cd'].astype(str)

In [12]:
# Bring together financial data and model data
scoring_df = filtered_df.merge(data_df[["admit_id"] + features],
on="admit_id", how="left",
suffixes=("","_model")
)

In [13]:
# Sanity Checks
# Be sure we have all the expected columns
missing_cols = [col for col in features if col not in scoring_df.columns]
print("Missing feature columns:", missing_cols)
# Check for rows with missing data
missing_feature_rows = scoring_df[features].isna().any(axis=1).sum()
print("Rows with missing model features:", missing_feature_rows)

Missing feature columns: []
Rows with missing model features: 0


### Score Admissions
Each admission is scored using the saved CAR024 model. The predicted probability represents the model-estimated likelihood that CAR024 should be present. The validation-selected threshold of 0.23 is then applied to create a binary review flag.

In [14]:
# Create new scoring dataframe
X_score=scoring_df[features].copy()

In [15]:
# Predict the probability of CAR024 given the inputs
scoring_df["y_prob"]=model.predict_proba(X_score)[:,1]

In [16]:
# Add binary prediction using the optimized threshold
scoring_df["y_pred"] = (scoring_df["y_prob"] >= threshold).astype(int)

In [17]:
# Calculate model performance metrics at the selected operating threshold
from sklearn.metrics import precision_score, recall_score

precision = precision_score(scoring_df["target_flag"], scoring_df["y_pred"])
recall = recall_score(scoring_df["target_flag"], scoring_df["y_pred"])
flag_rate = scoring_df["y_pred"].mean()

In [18]:
# Sanity Checks
print("Threshold:",threshold)
print("Predicted Positives:",scoring_df["y_pred"].sum())
print(f"Flag Rate: {flag_rate:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall: {recall:.6f}")
scoring_df["y_prob"].describe()

Threshold: 0.23
Predicted Positives: 69297
Flag Rate: 0.123676
Precision: 0.440495
Recall: 0.606220


count    560311.000000
mean          0.089720
std           0.143132
min           0.000481
25%           0.012288
50%           0.028651
75%           0.088840
max           0.936885
Name: y_prob, dtype: float64

Interpretation:<br>
The model loaded correctly and the threshold was applied correctly<br>
Mean Probability = 0.0897 : The average predicted probability of CAR024 is ~9% <br>
Median Probability = 0.0287 : Half of the admissions have a predicted probability below ~3% <br>
Max Probability = 0.9369 : Some admissions are highly CAR024-like

In [19]:
# One more sanity check: Compare predicted vs actual prevalence
actual_rate = scoring_df["target_flag"].mean()
print("Actual Prevalence:",f"{actual_rate:.6%}")
print("Predicted Prevalence:",f"{scoring_df['y_pred'].mean():.6%}")


Actual Prevalence: 8.986616%
Predicted Prevalence: 12.367596%


### Scoring Validation Interpretation

The saved CAR024 model loaded successfully and produced a reasonable probability distribution. <br>
The selected threshold identifies a manageable review population relative to the full admission set, <br>
supporting the use of model probabilities for targeted review prioritization.

From a review workflow perspective:<br>
True CAR024 cases: 9% (Actual Prevalence)<br>
The model is flagging 12% of cases<br>
If precision is ~42%, then among flagged cases, 42% actually have CAR024; 58% do not<br>
<br>
Operationally:
Instead of reviewing all 560,311 cases, we review ~12% (predicted prevalence)<br>
and within that subset, we can expect to capture ~57% of true CAR024 cases

In [20]:
# Calculate Lift
lift = recall / flag_rate

print(f"Lift: {lift:.2f}x")

Lift: 4.90x


In [21]:
# Save the scored admissions as a new dataframe
scored_df=scoring_df.copy()

### Opportunity Identification
Opportunity cases are admissions where CAR024 was not recorded, but the model predicted that the encounter was sufficiently CAR024-like to warrant review.<br><br>
_**These cases should not be interpreted as confirmed missed revenue. <br>They represent admissions where the observed procedure coding differs from the model’s expected procedure pattern and therefore may warrant manual review.**_

In [22]:
# Identify potential financial opportunity admissions
scored_df["opportunity_case"] = (
(scored_df["target_flag"] == 0)
&
(scored_df["y_pred"] ==1)
).astype(int)


In [23]:
opportunity_df = scored_df[scored_df["opportunity_case"]==1].copy()
print("Opportunity Cases:",
      f"{len(opportunity_df):,}")

Opportunity Cases: 38,772


In [24]:
# Calculate the spread of opportunity cases across DRGs
opportunity_drg_distribution = (
    opportunity_df
    .groupby("drg_cd")
    .agg(
        opportunity_cases=("admit_id", "count"),
        avg_probability=("y_prob", "mean"),
        median_delta_charge=("delta_charge", "median")
    )
    .reset_index()
)

opportunity_drg_distribution

,drg_cd,opportunity_cases,avg_probability,median_delta_charge
0,139,1076,0.319706,52420.5
1,194,7365,0.351207,59986.0
2,383,801,0.312856,26995.0
3,469,9302,0.368111,61283.0
4,710,20228,0.395336,98751.0


Expected financial opportunity is calculated as model probability multiplied by the DRG-level median charge delta. This should be interpreted as a benchmark-based review prioritization metric, not guaranteed recoverable revenue.

In [25]:
# Expected Financial Opportunity = Probability x Delta Charge
opportunity_df["expected_fin_opportunity"] = (
    opportunity_df["y_prob"] * opportunity_df["delta_charge"])


In [26]:
print("Average Opportunity per Case:",
      f"${opportunity_df['expected_fin_opportunity'].mean():,.2f}")

print("Median Opportunity per Case:",
      f"${opportunity_df['expected_fin_opportunity'].median():,.2f}")

Average Opportunity per Case: $30,421.47
Median Opportunity per Case: $28,049.04


IMPORTANT NOTE about the above numbers:<br>
A 'Total Opportunity' number is not calculated because we are using a national dataset. The intent is to demonstrate a scalable framework that can estimate case-level opportunities and prioritize records for review. The national dataset is being used to learn patterns and benchmark opportunity, not to estimate a national total financial opportunity.



In [27]:
# Calculate the potential opportunity at the DRG level
# This tells us which cases are worth reviewing first
drg_case_opportunity_summary = (
    opportunity_df
    .groupby("drg_cd")
    .agg(
        opportunity_cases=("admit_id", "count"),
        median_case_opportunity=(
            "expected_fin_opportunity", "median"
        ),
        avg_case_opportunity=(
            "expected_fin_opportunity", "mean"
        )
    )
    .reset_index()
)

drg_case_opportunity_summary

,drg_cd,opportunity_cases,median_case_opportunity,avg_case_opportunity
0,139,1076,15710.016461,16759.153124
1,194,7365,19484.406525,21067.522142
2,383,801,7552.152659,8445.545956
3,469,9302,20760.496855,22558.916985
4,710,20228,35520.520771,39039.858420


### Opportunity Tiering

To make the case-level opportunity estimates easier to interpret, expected opportunity is grouped into dollar-value tiers. These tiers help show whether the review queue is mostly made up of small, moderate, or high-value admissions.

In [28]:
# Create Opportunity Tiers or Buckets
opportunity_df["opportunity_tier"] = pd.cut(
    opportunity_df["expected_fin_opportunity"],
    bins=[0, 5000, 10000, 25000, 50000, np.inf],
    labels=[
        "<$5K",
        "$5K–10K",
        "$10K–25K",
        "$25K–50K",
        "$50K+"
    ]
)

### Tier Summary

In [29]:
# Summarize opportunity cases by expected dollar tier
tier_summary = (
    opportunity_df
    .groupby("opportunity_tier",
    observed=True)
    .agg(
        cases=("admit_id", "count"),
        median_opportunity=(
            "expected_fin_opportunity", "median"
        ),
        avg_opportunity=(
            "expected_fin_opportunity", "mean"
        )
    )
    .reset_index()
)

tier_summary

,opportunity_tier,cases,median_opportunity,avg_opportunity
0,$5K–10K,639,7134.651083,7426.580009
1,$10K–25K,14781,18395.692699,18814.804250
2,$25K–50K,19516,32674.157110,33996.878562
3,$50K+,3836,59897.034312,60784.944031


### Top-N Review Analysis
The Top-N analysis simulates constrained review capacity. Rather than reviewing every flagged case, this shows what the highest-ranked subset of cases looks like when prioritized by expected opportunity.

In [30]:
# Sort opportunity cases from highest to lowest expected opportunity.
# This allows the Top-N analysis to simulate a prioritized review queue.
opportunity_df = opportunity_df.sort_values(
    by="expected_fin_opportunity",
    ascending=False
).reset_index(drop=True)

review_levels = [0.01, 0.05, 0.10, 0.25]

top_review_summary = []

for pct in review_levels:

    n_cases = int(len(opportunity_df) * pct)

    subset = opportunity_df.head(n_cases)

    top_review_summary.append({
        "Review Queue":
            f"Top {int(pct*100)}%",

        "Cases Reviewed":
            n_cases,

        "Median Case Opportunity":
            subset["expected_fin_opportunity"].median(),

        "Average Case Opportunity":
            subset["expected_fin_opportunity"].mean(),

        "Total Expected Opportunity":
            subset["expected_fin_opportunity"].sum()
    })

top_review_df = pd.DataFrame(
    top_review_summary
)

top_review_df

,Review Queue,Cases Reviewed,Median Case Opportunity,Average Case Opportunity,Total Expected Opportunity
0,Top 1%,387,74141.463411,74989.833385,2.902107e+07
1,Top 5%,1938,65924.181648,66890.388064,1.296336e+08
2,Top 10%,3877,59741.884756,60669.784463,2.352168e+08
3,Top 25%,9693,46491.256868,49589.227383,4.806684e+08


### Threshold Sensitivity Analysis

The selected operating threshold balances precision and recall, but different organizations may prefer a more aggressive or more conservative review strategy depending on staffing, audit priorities, or tolerance for false positives.

This section evaluates how changes in the review threshold affect the size of the review queue and the expected case-level opportunity.

In [31]:
# Threshold Sensitivity Analysis
# Compares a more aggressive, selected, and more conservative review threshold.

threshold_scenarios = [
    {"Scenario": "Aggressive", "Threshold": 0.16},
    {"Scenario": "Selected Operating Threshold", "Threshold": threshold},
    {"Scenario": "Conservative", "Threshold": 0.30}
]

sensitivity_rows = []

for scenario in threshold_scenarios:
    scenario_name = scenario["Scenario"]
    t = scenario["Threshold"]

    temp_df = scored_df.copy()

    # Apply threshold
    temp_df["scenario_y_pred"] = (
        temp_df["y_prob"] >= t
    ).astype(int)

    # Opportunity case: target absent but model flags for review
    temp_df["scenario_opportunity_case"] = (
        (temp_df["target_flag"] == 0)
        &
        (temp_df["scenario_y_pred"] == 1)
    ).astype(int)

    scenario_opportunity_df = temp_df[
        temp_df["scenario_opportunity_case"] == 1
    ].copy()

    # Expected financial opportunity = probability x DRG-level median delta
    scenario_opportunity_df["expected_fin_opportunity"] = (
        scenario_opportunity_df["y_prob"]
        *
        scenario_opportunity_df["delta_charge"]
    )

    sensitivity_rows.append({
        "Scenario": scenario_name,
        "Threshold": t,
        "Cases Flagged": temp_df["scenario_y_pred"].sum(),
        "Flag Rate": temp_df["scenario_y_pred"].mean(),
        "Opportunity Cases": scenario_opportunity_df.shape[0],
        "Opportunity Rate": temp_df["scenario_opportunity_case"].mean(),
        "Median Case Opportunity": scenario_opportunity_df["expected_fin_opportunity"].median(),
        "Average Case Opportunity": scenario_opportunity_df["expected_fin_opportunity"].mean()
    })

threshold_sensitivity_df = pd.DataFrame(sensitivity_rows)

threshold_sensitivity_df.style.format({
    "Threshold": "{:.6f}",
    "Cases Flagged": "{:,.0f}",
    "Flag Rate": "{:.3%}",
    "Opportunity Cases": "{:,.0f}",
    "Opportunity Rate": "{:.3%}",
    "Median Case Opportunity": "${:,.2f}",
    "Average Case Opportunity": "${:,.2f}"
})

,Scenario,Threshold,Cases Flagged,Flag Rate,Opportunity Cases,Opportunity Rate,Median Case Opportunity,Average Case Opportunity
0,Aggressive,0.160000,"92,331",16.479%,"57,444",10.252%,"$21,206.81","$24,639.03"
1,Selected Operating Threshold,0.230000,"69,297",12.368%,"38,772",6.920%,"$28,049.04","$30,421.47"
2,Conservative,0.300000,"52,339",9.341%,"26,220",4.680%,"$33,137.88","$35,466.10"


**Interpretation**<br>
The lower threshold casts a wider net and identifies more possible opportunities, but the average value per case declines. <br>
Why?<br>
We begin reviewing more marginal cases with lower model confidence.<br>
Operationally: higher workload ==> lower value density.<br><br>
The median threshold is the balance point. <br>
We are still capturing a substantial review population while materially improving expected value per case.<br><br>
The higher threshold produces a smaller, more concentrated review queue. There are fewer cases, but with an increased median case opportunity. <br>
Operationally: Best when staffing is contrained

### Highest-Value Case Profile

After selecting an operating threshold, the final step is to examine the highest-value subset of opportunity cases. This helps identify which DRGs contribute most to the prioritized review queue and where review effort may be most impactful.

In [32]:
# Profile the highest-value opportunity cases by DRG
# Here, "highest value" is defined as the top 10% of opportunity cases
# ranked by expected financial opportunity.

top_10 = opportunity_df.head(
    int(len(opportunity_df) * 0.10)
).copy()

top_10_drg_summary = (
    top_10
    .groupby("drg_cd")
    .agg(
        high_value_cases=("admit_id", "count"),
        median_case_opportunity=("expected_fin_opportunity", "median"),
        avg_case_opportunity=("expected_fin_opportunity", "mean"),
        total_expected_opportunity=("expected_fin_opportunity", "sum")
    )
    .reset_index()
)

top_10_drg_summary["pct_of_top_10"] = (
    top_10_drg_summary["high_value_cases"]
    /
    top_10_drg_summary["high_value_cases"].sum()
)

top_10_drg_summary = top_10_drg_summary.sort_values(
    by="total_expected_opportunity",
    ascending=False
)

top_10_drg_summary.style.format({
    "high_value_cases": "{:,.0f}",
    "median_case_opportunity": "${:,.2f}",
    "avg_case_opportunity": "${:,.2f}",
    "total_expected_opportunity": "${:,.2f}",
    "pct_of_top_10": "{:.2%}"
})

,drg_cd,high_value_cases,median_case_opportunity,avg_case_opportunity,total_expected_opportunity,pct_of_top_10
1,710,"3,874","$59,756.16","$60,677.22","$235,063,556.48",99.92%
0,469,3,"$51,256.38","$51,065.96","$153,197.88",0.08%


### In Conclusion

The financial analysis is intentionally framed at the case level. Because the source data is national, aggregate opportunity totals should not be interpreted as hospital-level financial impact. Instead, the value of the model is in prioritizing individual admissions for review based on predicted likelihood and benchmarked charge differences.

In practical terms, the model supports a targeted review workflow: rather than reviewing all admissions, a hospital could focus on a smaller group of high-priority cases where the expected opportunity is materially higher.